# Bu notebookda projede kullanılacak makine öğrenim süreçleri modeller ve searchler bulunmaktadır


vektörize edilmiş market databaseimiz için search meselesi


In [1]:
import joblib
import polars as pl
from rapidfuzz import fuzz
from sklearn.metrics.pairwise import cosine_similarity

def turkce_karakter_temizle(metin):
    harfler = {
        "ç": "c", "ş": "s", "ğ": "g", "ı": "i", "ö": "o", "ü": "u",
        "Ç": "c", "Ş": "s", "Ğ": "g", "İ": "i", "Ö": "o", "Ü": "u"
    }
    for tr, eng in harfler.items():
        metin = metin.replace(tr, eng)
    return metin.lower()

loaded_vectorizer = joblib.load('tfidf_vectorizer.pkl')
loaded_matrix = joblib.load('tfidf_matrix.pkl')
loaded_df = pl.read_parquet('cleaned_dataframe.parquet')

def hibrit_market_aramasi(malzeme_adi, top_n=3):
    temiz_malzeme = turkce_karakter_temizle(malzeme_adi)
    
    birlesik_malzeme = temiz_malzeme.replace(" ", "")
    
    vec_ayri = loaded_vectorizer.transform([temiz_malzeme])
    vec_birlesik = loaded_vectorizer.transform([birlesik_malzeme])
    
    skor_ayri = cosine_similarity(vec_ayri, loaded_matrix).flatten()
    skor_birlesik = cosine_similarity(vec_birlesik, loaded_matrix).flatten()
    
    if skor_birlesik.max() > skor_ayri.max():
        aktif_skorlar = skor_birlesik
        aktif_kelime = birlesik_malzeme
    else:
        aktif_skorlar = skor_ayri
        aktif_kelime = temiz_malzeme
        
    aday_indeksler = aktif_skorlar.argsort()[-20:][::-1]
    aday_listesi = []
    
    for idx in aday_indeksler:
        tfidf_skor = aktif_skorlar[idx]
        urun = loaded_df["ITEMNAME"][int(idx)]
        kat1 = loaded_df["CATEGORY1"][int(idx)]
        kat2 = loaded_df["CATEGORY2"][int(idx)]
        
        urun_temiz = turkce_karakter_temizle(urun)
        
        fuzzy_skor = fuzz.token_set_ratio(aktif_kelime, urun_temiz) / 100.0
        
        final_skor = (tfidf_skor * 0.3) + (fuzzy_skor * 0.7)
        
        aday_listesi.append({
            "urun": urun,
            "kat1": kat1,
            "kat2": kat2,
            "skor": final_skor,
            "detay": f"(TF-IDF: {tfidf_skor:.2f} | Fuzzy: {fuzzy_skor:.2f})"
        })
        
    aday_listesi = sorted(aday_listesi, key=lambda x: x["skor"], reverse=True)
    
    print(f"\n--- Girdi: '{malzeme_adi}' | Algoritmanın Kararı: '{aktif_kelime}' ---")
    for i in range(min(top_n, len(aday_listesi))):
        sonuc = aday_listesi[i]
        print(f"[{sonuc['skor']:.2f}] [{sonuc['kat1']} > {sonuc['kat2']}] {sonuc['urun']} {sonuc['detay']}")




hibrit_market_aramasi("sucuk dana")



--- Girdi: 'sucuk dana' | Algoritmanın Kararı: 'sucuk dana' ---
[0.94] [ET > ISLENMIS ET] AYTAC SUCUK 220 GR DANA GURME *8* (TF-IDF: 0.78 | Fuzzy: 1.00)
[0.93] [ET > ISLENMIS ET] AYTAC SUCUK 300 GR DANA CERKEZ *8* (TF-IDF: 0.77 | Fuzzy: 1.00)
[0.93] [ET > ISLENMIS ET] NAMET SUCUK 225GR DANA ETI (TF-IDF: 0.77 | Fuzzy: 1.00)


kısaca yukarda ne yaptığımı anlatayım bizim yemek tariflerinden çıkaracağımız malzemeleri market database'imizde bulmamız lazımdı bunun için search algoritma modellerine ihtiyacımız vardı ilk adım olarak tf-idf ile kelimelerimizin matematiğini çıkardık sonrasında kosinüs benzerliği ile basit bi algoritma kurdum ancak domates - domates salçası aratmasında aynı sonuçlar geldi çünkü ikisi de kosinüs olarak neredeyse aynı ama farklı şeyler bunun sonucunda oyuna rapidfuzz geldi ve cümle uzunluklarını kattı sonra salça aratması yaptım alakasız şeyler geldi çünkü türkçe karakterleri anlamıyordu algoritma basit bi türkçe karakter dönüşümü ekledim bu sorun da böyle çözüldü sonrasında kara biber ve karabiber araması yaptım apayrı sonuçlar çıktı sonrasında boşluk için birleşik arama da ekledim